# Week 9: Natural Language Processing - Text, Sentiment, Entities, and Speech

Natural language processing (NLP) is getting computers to work with human language. Models need numbers, so the core problem is turning words into numbers and then using the tools we already know (train/test split, classifiers, precision and recall).

**Key terms for today**
- Corpus: a collection of text documents
- Token: a unit of text, usually a word
- Stop words: very common words (the, a, of) that carry little meaning
- Bag of words: represent a document by counting its words, ignoring word order
- TF-IDF: a weighted version of bag of words that gives more weight to words that are distinctive and less to words that appear everywhere
- Sentiment analysis: classifying text as positive or negative
- Named entity recognition (NER): finding names of people, organizations, places, dates and amounts in text
- Speech recognition (speech to text): turning audio into a transcript

Modern NLP uses large neural networks (the technology behind chatbots like Claude and ChatGPT). Today we start with the fundamentals those systems build on.

**Roadmap**
1. Text to numbers
2. A sentiment classifier
3. Entity recognition
4. Speech: from sound to text
5. In-class activity (IC-8)

Same format as before: a demo that runs as-is, then Your Turn on your own dataset. **The Your Turn cells have a blank (set to None) that you must fill in before the cell will run.** Each has a TODO comment and suggested values to try.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

nltk.download('movie_reviews', quiet=True)
nltk.download('sentence_polarity', quiet=True)

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 80)
print('Libraries loaded.')

---
## Step 0 - Load the Text

**Demo dataset:** 2,000 movie reviews labeled positive or negative (long, several paragraphs each).

In [ ]:
from nltk.corpus import movie_reviews

texts_d = [movie_reviews.raw(f) for f in movie_reviews.fileids()]
labels_d = np.array([1 if f.startswith('pos') else 0 for f in movie_reviews.fileids()])   # 1 = positive

print('Reviews:', len(texts_d), ' Share positive:', labels_d.mean())
print('Average words per review:', int(np.mean([len(t.split()) for t in texts_d])))
print('\nExample (first 300 characters), label =', labels_d[0])
print(texts_d[0][:300])

### Your Turn - Load your text

**Your dataset: 10,662 short movie review snippets**, each one sentence, labeled positive or negative. Much shorter than the demo reviews.

TODO: set `SAMPLE_ROW` to read any snippet (a number from 0 to 10661). The file is ordered by label: the first half is negative and the second half is positive, so try rows from both halves.

In [ ]:
from nltk.corpus import sentence_polarity

texts_y, labels_y = [], []
for category in sentence_polarity.categories():
    for sent in sentence_polarity.sents(categories=category):
        texts_y.append(' '.join(sent))
        labels_y.append(1 if category == 'pos' else 0)
labels_y = np.array(labels_y)

SAMPLE_ROW = None   # TODO: a number from 0 to 10661
assert SAMPLE_ROW is not None, 'Set SAMPLE_ROW above first, then re-run this cell.'

print('Snippets:', len(texts_y), ' Share positive:', labels_y.mean())
print('Average words per snippet:', int(np.mean([len(t.split()) for t in texts_y])))
print(f'\nRow {SAMPLE_ROW}, label = {labels_y[SAMPLE_ROW]} (1 = positive):')
print(texts_y[SAMPLE_ROW])

**Your answers:**
1. Read a few snippets from each half. Can you always tell the sentiment from the words alone? Find one that is hard.
2. The data is sorted by label. Why would that be a problem if we split it by taking the first 80 percent as training data?

---
## Step 1 - Text to Numbers

A classifier cannot read words, so we convert each document into a row of numbers:

1. Split the text into tokens (lowercase words)
2. Build a vocabulary: every distinct word across the documents
3. For each document, count how many times each vocabulary word appears (bag of words)

TF-IDF adjusts those counts: a word that appears in every document (like "movie") gets a low weight, and a word that is rare and distinctive (like "terrible") gets a high weight.

In [ ]:
tiny = ['The movie was great', 'The movie was terrible', 'Great acting, great story']
labels_tiny = ['doc 1', 'doc 2', 'doc 3']

cv = CountVectorizer()
counts = cv.fit_transform(tiny)
print('Bag of words (word counts):')
display(pd.DataFrame(counts.toarray(), columns=cv.get_feature_names_out(), index=labels_tiny))

tf = TfidfVectorizer()
weights = tf.fit_transform(tiny)
print('TF-IDF weights:')
display(pd.DataFrame(weights.toarray(), columns=tf.get_feature_names_out(), index=labels_tiny))

Each document is now a row of numbers, and word order is gone ("dog bites man" and "man bites dog" would look identical). That loss of order will matter later.

### Your Turn - Build your own bag of words

TODO: set two values.
- `my_sentences`: a list of 3 short sentences you write
- `REMOVE_STOP`: `True` to drop stop words like "the" and "was", or `False` to keep them

In [ ]:
my_sentences = None   # TODO: a list of 3 sentences, e.g. ['I love this pizza', 'The pizza was cold', 'Love the service']
REMOVE_STOP = None    # TODO: True or False
assert my_sentences is not None and REMOVE_STOP is not None, 'Set my_sentences and REMOVE_STOP above first, then re-run this cell.'

cv_y = CountVectorizer(stop_words='english' if REMOVE_STOP else None)
counts_y = cv_y.fit_transform(my_sentences)
display(pd.DataFrame(counts_y.toarray(), columns=cv_y.get_feature_names_out(),
                     index=[f'doc {i + 1}' for i in range(len(my_sentences))]))
print('Vocabulary size:', len(cv_y.get_feature_names_out()))

**Your answers:**
1. How did the vocabulary change when you switched `REMOVE_STOP`? Which words disappeared?
2. Write two sentences with opposite meanings that produce almost the same word counts. What does that tell you about bag of words?

---
## Step 2 - A Sentiment Classifier

Now the same workflow as earlier weeks. Turn text into TF-IDF numbers, train a logistic regression, and evaluate it on held-out text with accuracy, the confusion matrix, precision, recall and F1.

Two details. First, the vectorizer learns its vocabulary from the training text only. If it saw the test text, test information would leak into training (the same leakage idea as before). Second, we remove stop words so the most important words the model learns are meaningful ones.

In [ ]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    texts_d, labels_d, test_size=0.2, random_state=42, stratify=labels_d)

sent_model_d = make_pipeline(TfidfVectorizer(max_features=5000, stop_words='english'), LogisticRegression(max_iter=2000))
sent_model_d.fit(X_train_d, y_train_d)
preds_d = sent_model_d.predict(X_test_d)

print('Test accuracy:', round(accuracy_score(y_test_d, preds_d), 3))
ConfusionMatrixDisplay.from_predictions(y_test_d, preds_d, display_labels=['Negative', 'Positive'], cmap='Blues')
plt.title('Demo: sentiment confusion matrix')
plt.show()
print(classification_report(y_test_d, preds_d, target_names=['Negative', 'Positive']))

In [ ]:
def top_words(model, n=8):
    vec, clf = model.steps[0][1], model.steps[1][1]
    words = np.array(vec.get_feature_names_out())
    order = np.argsort(clf.coef_[0])
    return {'most negative words': list(words[order[:n]]), 'most positive words': list(words[order[-n:]][::-1])}

def positive_probability(model, text):
    return round(float(model.predict_proba([text])[0][1]), 2)

for key, words in top_words(sent_model_d).items():
    print(key + ':', words)

print()
for s in ['A wonderful, moving film with great acting', 'A boring, awful waste of time',
          'The plot was not bad at all', 'This movie is not good']:
    print(positive_probability(sent_model_d, s), s)

The model learned which words signal each class, which is why it is explainable. (A few words like "life" or "war" reflect quirks of this particular set of reviews rather than real sentiment.) The probabilities show how sure it is that a sentence is positive (1.0 is certain). It handles the clear cases, but "not bad at all" fools it: it sees the word "bad" and ignores the "not" in front of it. Bag of words has no sense of word order.

### Your Turn - Train and test your own sentiment model

TODO: set two values.
- `MAX_FEATURES`: how many words the model may use (try 50, 500, 2000, 10000)
- `NGRAM`: `(1, 1)` for single words, or `(1, 2)` for single words and two-word phrases

In [ ]:
MAX_FEATURES = None   # TODO: vocabulary size. Try 50, 500, 2000, 10000.
NGRAM = None          # TODO: (1, 1) or (1, 2)
assert MAX_FEATURES is not None and NGRAM is not None, 'Set MAX_FEATURES and NGRAM above first, then re-run this cell.'

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    texts_y, labels_y, test_size=0.2, random_state=42, stratify=labels_y)

sent_model_y = make_pipeline(TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM, stop_words='english'),
                             LogisticRegression(max_iter=2000))
sent_model_y.fit(X_train_y, y_train_y)
preds_y = sent_model_y.predict(X_test_y)

print('Test accuracy:', round(accuracy_score(y_test_y, preds_y), 3))
ConfusionMatrixDisplay.from_predictions(y_test_y, preds_y, display_labels=['Negative', 'Positive'], cmap='Blues')
plt.title('Your model: sentiment confusion matrix')
plt.show()
print(classification_report(y_test_y, preds_y, target_names=['Negative', 'Positive']))

for key, words in top_words(sent_model_y).items():
    print(key + ':', words)

print('\nThree snippets the model got wrong:')
wrong = np.where(preds_y != np.array(y_test_y))[0][:3]
for i in wrong:
    print(f'  true = {y_test_y[i]}, predicted = {preds_y[i]}: {X_test_y[i]}')

In [ ]:
MY_TEXT = None   # TODO: write a sentence. Try one with "not", one with sarcasm, one with mixed feelings.
assert MY_TEXT is not None, 'Set MY_TEXT above first, then re-run this cell.'

print('Probability the model thinks it is positive:', positive_probability(sent_model_y, MY_TEXT))

**Your answers:**
1. What happened to accuracy as you changed `MAX_FEATURES`? Why does a tiny vocabulary hurt?
2. Do the most positive and most negative words make sense? Is any word surprising?
3. Did your own sentence fool the model? If so, why?

---
## Step 3 - Entity Recognition

Named entity recognition (NER) finds and labels the "who, where, when, how much" in text. Common entity types:

- PERSON: people
- ORG: companies, agencies, institutions
- GPE: countries, cities, states
- DATE and TIME: dates and times
- MONEY: monetary amounts

We use spaCy, a library with a small pretrained English model. We do not train anything here. The model was trained on large amounts of labeled text and predicts a label for each span of words it recognizes, using the surrounding words as clues. Like any model, it makes mistakes.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp = spacy.load('en_core_web_sm')

def find_entities(text):
    doc = nlp(text)
    return pd.DataFrame([{'text': e.text, 'type': e.label_, 'meaning': spacy.explain(e.label_)} for e in doc.ents])

for s in ['Apple opened a new store in Houston on Monday.',
          'Dr. Maria Lopez joined the University of Houston in 2021 with a $2 million grant.',
          'Tesla shares rose 5 percent after Elon Musk spoke in Austin.']:
    print(s)
    display(find_entities(s))

### Your Turn - Find entities in your own text

TODO: set two values.
- `TEXT`: a sentence you write or paste (a news headline works well)
- `KEEP`: `'ALL'` to show everything, or one type such as `'PERSON'`, `'ORG'`, `'GPE'`, `'DATE'`, `'MONEY'`

Try to break it: all lowercase text, misspelled names, or a word with two meanings like "Apple" the fruit versus the company.

In [ ]:
TEXT = None   # TODO: a sentence with names, places, dates or money
KEEP = None   # TODO: 'ALL' or one entity type such as 'ORG'
assert TEXT is not None and KEEP is not None, 'Set TEXT and KEEP above first, then re-run this cell.'

ents = find_entities(TEXT)
if KEEP != 'ALL' and not ents.empty:
    ents = ents[ents['type'] == KEEP]

print(TEXT)
display(ents)
print('Entities shown:', len(ents))

**Your answers:**
1. Did the model find everything you expected? Describe one miss or one wrong label.
2. What happened when you changed capitalization or used a word with two meanings? What does that say about the clues the model relies on?

---
## Step 4 - Speech: From Sound to Text

Speech recognition (speech to text) turns audio into a transcript. The basic pipeline:

1. Record the audio and slice it into short time frames
2. Convert each frame into numbers that describe its sound (a spectrogram)
3. A trained neural network maps those sound patterns to words
4. Output a transcript, which other NLP steps (sentiment, entities, search) can then use

Text to speech does the reverse. In practice most applications call a cloud speech API (Week 7) rather than training their own, since these models need thousands of hours of recorded speech.

Speech systems struggle with accents, background noise, similar-sounding words (homophones) and unusual names. We measure them with **word error rate (WER)**:

WER = (substitutions + deletions + insertions) / number of words in the correct transcript

Lower is better, 0 is perfect. We skip audio files here and work with transcripts, since the evaluation idea is the same.

In [ ]:
def word_error_rate(reference, hypothesis):
    ref, hyp = reference.lower().split(), hypothesis.lower().split()
    d = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)
    d[:, 0] = range(len(ref) + 1)
    d[0, :] = range(len(hyp) + 1)
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)
    return d[len(ref), len(hyp)] / len(ref)

reference = 'please send the report to Houston by Friday'
heard_d = 'please send the report to house in by Friday'

print('Reference :', reference)
print('Recognizer:', heard_d)
print('Word error rate:', round(word_error_rate(reference, heard_d), 2))

print('\nEntities in the correct transcript:')
display(find_entities(reference))
print('Entities in the recognizer output:')
display(find_entities(heard_d))

The word error rate is only 0.25, yet the city is gone: the entity recognizer cannot find "Houston" in a transcript that says "house in". Errors in an early stage carry forward, so a pipeline is only as good as its weakest stage. WER also treats every word equally, even though some words (names, places, amounts) matter much more than others.

### Your Turn - Mishear a sentence

The correct transcript is given. TODO: set `heard` to what a speech recognizer might wrongly transcribe. Change, drop or add some words. Try a version with one small error and another with many.

In [ ]:
reference_y = 'the meeting with Sarah Johnson in Chicago starts on Tuesday'
heard = None   # TODO: type a mistranscribed version of the sentence above
assert heard is not None, 'Set heard above first, then re-run this cell.'

print('Reference :', reference_y)
print('Recognizer:', heard)
print('Word error rate:', round(word_error_rate(reference_y, heard), 2))

print('\nEntities in the correct transcript:')
display(find_entities(reference_y))
print('Entities in the recognizer output:')
display(find_entities(heard))

**Your answers:**
1. What was your WER? Which errors hurt the entity recognizer, and which did not?
2. Why might two transcripts with the same WER still differ in how useful they are?

---
## Step 5 - In-Class Activity (IC-9): Where Does Language AI Break?

Three short tasks. Use the Your Turn cells above and change one setting at a time. Write one or two sentences for each.

**Task 1 (about 3 minutes).** In Step 2, set `MAX_FEATURES = 2000` and `NGRAM = (1, 1)` and run the cell. Record the accuracy and the two most negative words. Then change `MAX_FEATURES` to 50 and run again. What happened to accuracy, and why?

**Task 2 (about 3 minutes).** Use `MY_TEXT` to write one sentence that you think will fool the sentiment model (negation, sarcasm, or mixed feelings). Did it work? Explain why bag of words fails on it.

**Task 3 (about 2 minutes).** In Step 4, enter a mistranscribed sentence and compare the entities found. If a company used speech recognition followed by entity recognition to log customer calls, what could go wrong?

**Your answers:**

Task 1:

Task 2:

Task 3:

**To submit for the in-class activity:** upload a screenshot of your Step 2 confusion matrix along with your three written answers to the IC-8 module.

---
# Assignment 9 - Build, Break, and Explain a Text Classifier

Run the notebook from the top with your chosen settings first. Write your own explanations; answers that only restate numbers will not get full credit.

## Task 1 - Feature experiment

Compare vocabulary sizes and single words versus word pairs on the same train/test split.

TODO: set `max_features_options` to a list of at least three sizes (for example `[100, 1000, 10000]`).

In [ ]:
max_features_options = None   # TODO: a list of at least 3 sizes, e.g. [100, 1000, 10000]
assert max_features_options is not None and len(max_features_options) >= 3, 'Set max_features_options to a list of at least 3 values first.'

rows = []
for ngram in [(1, 1), (1, 2)]:
    for mf in max_features_options:
        model = make_pipeline(TfidfVectorizer(max_features=mf, ngram_range=ngram, stop_words='english'), LogisticRegression(max_iter=2000))
        model.fit(X_train_y, y_train_y)
        rows.append({'words used': str(ngram), 'max_features': mf, 'accuracy': accuracy_score(y_test_y, model.predict(X_test_y))})

display(pd.DataFrame(rows).pivot(index='max_features', columns='words used', values='accuracy'))

**Task 1 answer.** Which setting was best? Did word pairs (1, 2) help? Is there a point where more vocabulary stops helping? Why might that be?

**Your answer:**

## Task 2 - Break the model

Write sentences you expect the sentiment model to get wrong, then check.

In [ ]:
test_sentences = None   # TODO: a list of at least 3 sentences you expect might fool the model (negation, sarcasm, mixed)
assert test_sentences is not None and len(test_sentences) >= 3, 'Set test_sentences to a list of at least 3 sentences first.'

for s in test_sentences:
    print(positive_probability(sent_model_y, s), s)

**Task 2 answer.** Which sentences fooled the model and why? Group them by type of failure (negation, sarcasm, and so on). What would a model need to understand to get them right?

**Your answer:**

## Task 3 - Reflection

In 6 to 8 sentences:
1. Why is the vectorizer fit on the training text only? What would go wrong if it saw the test text first?
2. A company wants to analyze customer reviews and call recordings. Would you build your own models or use a cloud NLP and speech API? Use what you learned about cost, privacy and accuracy in Week 7.
3. Sentiment models make errors that matter differently depending on the use. Give one use where a wrong "positive" is worse than a wrong "negative", and say why.

**Your answer:**